# 🧠 3D Brain Tumor Sub-Region Segmentation with SwinUNETR on BraTS

**Research Lab:** ARCHE Research Lab  
**Architecture:** 3D SwinUNETR (Swin Transformer for Medical Image Segmentation)  
**Benchmark:** BraTS 2020 (Multimodal 3D MRI: FLAIR, T1, T1ce, T2)  
**Framework:** PyTorch & MONAI (Medical Open Network for AI)

---

## 📋 Scientific Rationale
Automated 3D brain tumor segmentation is crucial for neuro-surgical planning and radiotherapy. 
BraTS defines three nested clinical sub-regions:
- **Tumor Core (TC):** Necrotic & non-enhancing core (label 1) + Enhancing tumor (label 4)
- **Whole Tumor (WT):** Necrotic core (1) + Edema (2) + Enhancing tumor (4)
- **Enhancing Tumor (ET):** Active gadolinium-enhancing rim (label 4)

This notebook provides a complete PyTorch/MONAI implementation of 3D SwinUNETR with full-volume sliding window inference.

In [ ]:
# Cell 1: Environment and Library Verification
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import monai
from monai.networks.nets import SwinUNETR
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    CropForegroundd,
    RandSpatialCropd,
    RandFlipd,
    NormalizeIntensityd,
    RandShiftIntensityd,
    MapTransform,
    EnsureTyped,
    ConcatItemsd,
)

print(f"PyTorch Version : {torch.__version__}")
print(f"MONAI Version   : {monai.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
total_device = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f"Active Device   : {device} ({total_device} GPUs)")

In [ ]:
# Cell 2: Deterministic Seeding
monai.utils.set_determinism(seed=101)
torch.manual_seed(101)
np.random.seed(101)
print("Deterministic seed set to: 101")

In [ ]:
# Cell 3: Multi-Label Clinical Target Mapping
class ConvertToMultiChannelBratsd(MapTransform):
    """Converts integer BraTS labels to 3-channel binary multi-label target: TC, WT, ET."""
    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            label = d[key]
            if isinstance(label, torch.Tensor):
                label = label.cpu().numpy()
            if label.ndim == 4 and label.shape[0] == 1:
                label = label[0]

            tc = (label == 1) | (label == 4)
            wt = (label == 1) | (label == 2) | (label == 4)
            et = (label == 4)
            d[key] = np.stack([tc, wt, et], axis=0).astype(np.float32)
        return d

print("ConvertToMultiChannelBratsd defined successfully.")

In [ ]:
# Cell 4: MONAI Data Augmentation Pipelines
modality_keys = ["flair", "t1", "t1ce", "t2"]

train_transform = Compose([
    LoadImaged(keys=modality_keys + ["label"]),
    EnsureChannelFirstd(keys=modality_keys + ["label"]),
    ConcatItemsd(keys=modality_keys, name="image", dim=0),
    ConvertToMultiChannelBratsd(keys=["label"]),
    CropForegroundd(keys=["image", "label"], source_key="image", k_divisible=[96, 96, 96], allow_smaller=True),
    RandSpatialCropd(keys=["image", "label"], roi_size=(96, 96, 96), random_size=False),
    RandFlipd(keys=["image", "label"], spatial_axis=0, prob=0.5),
    RandFlipd(keys=["image", "label"], spatial_axis=1, prob=0.5),
    RandFlipd(keys=["image", "label"], spatial_axis=2, prob=0.5),
    NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
    RandShiftIntensityd(keys=["image"], offsets=0.10, prob=1.0),
    EnsureTyped(keys=["image", "label"]),
])

val_transform = Compose([
    LoadImaged(keys=modality_keys + ["label"]),
    EnsureChannelFirstd(keys=modality_keys + ["label"]),
    ConcatItemsd(keys=modality_keys, name="image", dim=0),
    ConvertToMultiChannelBratsd(keys=["label"]),
    NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
    EnsureTyped(keys=["image", "label"]),
])
print("Transforms compiled successfully.")

In [ ]:
# Cell 5: Dataset Loader & Case Dict Discovery
def build_case_dicts(training_root):
    case_dirs = sorted(glob.glob(os.path.join(training_root, "BraTS20_Training_*")))
    if not case_dirs:
        case_dirs = sorted([os.path.join(training_root, d) for d in os.listdir(training_root) if os.path.isdir(os.path.join(training_root, d))])
    data_dicts = []
    for cdir in case_dirs:
        cid = os.path.basename(cdir)
        d = {
            "flair": os.path.join(cdir, f"{cid}_flair.nii"),
            "t1": os.path.join(cdir, f"{cid}_t1.nii"),
            "t1ce": os.path.join(cdir, f"{cid}_t1ce.nii"),
            "t2": os.path.join(cdir, f"{cid}_t2.nii"),
            "label": os.path.join(cdir, f"{cid}_seg.nii"),
        }
        if all(os.path.exists(p) for p in d.values()):
            data_dicts.append(d)
    return data_dicts

class BratsNiftiDataset(Dataset):
    def __init__(self, data_dicts, transform):
        self.data_dicts = data_dicts
        self.transform = transform
    def __len__(self):
        return len(self.data_dicts)
    def __getitem__(self, idx):
        sample = self.transform(self.data_dicts[idx])
        return sample["image"], sample["label"]
print("BratsNiftiDataset class ready.")

In [ ]:
# Cell 6: SwinUNETR Model Initialization
num_classes = 3  # TC, WT, ET
roi_size = (96, 96, 96)

model = SwinUNETR(
    in_channels=4,
    out_channels=num_classes,
    feature_size=48,
    use_checkpoint=True,
).to(device)

if total_device > 1:
    model = nn.DataParallel(model)

loss_fn = DiceCELoss(sigmoid=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
dice_metric = DiceMetric(include_background=True, reduction="mean", ignore_empty=True)
dice_metric_batch = DiceMetric(include_background=True, reduction="mean_batch", ignore_empty=True)

n_params = sum(p.numel() for p in model.parameters())
print(f"3D SwinUNETR Initialized -> Total Parameters: {n_params/1e6:.1f}M")

In [ ]:
# Cell 7: Full-Volume Sliding-Window Validation Function
def post_pred(logits):
    return (torch.sigmoid(logits) > 0.5).float()

def validate(model, loader):
    model.eval()
    dice_metric.reset()
    dice_metric_batch.reset()
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = sliding_window_inference(
                inputs=x,
                roi_size=roi_size,
                sw_batch_size=4 * total_device,
                predictor=model,
                overlap=0.5,
                mode="gaussian",
            )
            preds = post_pred(logits)
            dice_metric(y_pred=preds, y=y)
            dice_metric_batch(y_pred=preds, y=y)
    mean_dice = dice_metric.aggregate().item()
    per_class = dice_metric_batch.aggregate().cpu().numpy()
    return mean_dice, per_class
print("Validation engine ready.")

In [ ]:
# Cell 8: Reconstruct Clinical Integer Label Map
def reconstruct_integer_mask(segment):
    """Reconstructs integer label map (0, 1, 2, 4) from binary multi-label prediction."""
    prediction = np.zeros(segment.shape[:3], dtype='float32')
    prediction[segment[..., 1] == 1] = 2   # WT -> Peritumoral Edema
    prediction[segment[..., 0] == 1] = 1   # TC -> Necrotic / Core
    prediction[segment[..., 2] == 1] = 4   # ET -> Enhancing Tumor
    return prediction
print("Mask reconstruction function ready.")